In [1]:
!curl -L -A "Mozilla" -o "data/llama2.pdf" "https://arxiv.org/pdf/2307.09288.pdf"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   249  100   249    0     0    404      0 --:--:-- --:--:-- --:--:--   404
100 13.0M  100 13.0M    0     0  3502k      0  0:00:03  0:00:03 --:--:-- 4410k2790k      0  0:00:04  0:00:02  0:00:02 3890k


In [5]:
from llama_index.readers.file import PDFReader
from llama_index.readers.file import PyMuPDFReader
from pathlib import Path

loader = PyMuPDFReader()

documents = loader.load(file_path=Path("./data/llama2.pdf"))

In [6]:
# for better auto-mergining we need all the documents in single doc
# but loader loads each page in different doc hence we need to concat them in single doc

from llama_index.core import Document

doc_text = "\n\n".join([d.get_content() for d in documents])

docs = [Document(text=doc_text)]

In [7]:
from llama_index.core.node_parser import HierarchicalNodeParser

parser = HierarchicalNodeParser.from_defaults()

nodes = parser.get_nodes_from_documents(docs)

len(nodes)

1009

In [8]:
from llama_index.core.node_parser import get_leaf_nodes, get_root_nodes

leaf_node = get_leaf_nodes(nodes)
print(len(leaf_node))

root_nodes = get_root_nodes(nodes)
print(len(root_nodes))

783
47


In [9]:
import os
from dotenv import load_dotenv
from llama_index.llms.google_genai import GoogleGenAI

load_dotenv()

api_key = os.environ.get("GOOGLE_API_KEY")

llm = GoogleGenAI(model="gemini-2.0-flash", api_key=api_key)

In [11]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en")

In [12]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model

In [10]:
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core import StorageContext

docstore = SimpleDocumentStore()

docstore.add_documents(nodes)

storage_context = StorageContext.from_defaults(docstore=docstore)

In [13]:
from llama_index.core import VectorStoreIndex

base_index = VectorStoreIndex(
    leaf_node,
    storage_context=storage_context,
)

In [14]:
from llama_index.core.retrievers import AutoMergingRetriever

base_retriever = base_index.as_retriever(similarity_top_k=6)

retriever = AutoMergingRetriever(base_retriever, storage_context, verbose=True)

In [15]:
query_str = (
    "What could be the potential outcomes of adjusting the amount of safety"
    " data used in the RLHF stage?"
)

retrieved_nodes = retriever.retrieve(query_str)
base_nodes = base_retriever.retrieve(query_str)

> Merging 3 nodes into parent node.
> Parent node id: 2eb04faa-a424-4fb6-ad87-82e27f451747.
> Parent node text: We conduct RLHF by first collecting human preference data for safety similar to Section 3.2.2: an...



In [17]:
print(len(retrieved_nodes))
print(len(base_nodes))

4
6


In [19]:
from llama_index.core.response.notebook_utils import display_source_node

for node in retrieved_nodes:
    display_source_node(node, source_length=10000)

**Node ID:** 754e8c80-0739-4b73-9c4f-b49aed28d11a<br>**Similarity:** 0.8893585339004003<br>**Text:** 0
0.2
0.4
0.6
0.8
1.0
Helpfulness RM Score before Safety RLHF
0.0
0.2
0.4
0.6
0.8
1.0
Helpfulness RM Score after Safety RLHF
0
1000
0
1000
Figure 14: Impact of safety RLHF measured by reward model score distributions. Left: safety reward
model scores of generations on the Meta Safety test set. The clustering of samples in the top left corner
suggests the improvements of model safety.<br>

**Node ID:** 2eb04faa-a424-4fb6-ad87-82e27f451747<br>**Similarity:** 0.8892681687153344<br>**Text:** We conduct RLHF by first collecting human preference data for safety similar to Section 3.2.2: annotators
write a prompt that they believe can elicit unsafe behavior, and then compare multiple model responses to
the prompts, selecting the response that is safest according to a set of guidelines. We then use the human
preference data to train a safety reward model (see Section 3.2.2), and also reuse the adversarial prompts to
sample from the model during the RLHF stage.
Better Long-Tail Safety Robustness without Hurting Helpfulness
Safety is inherently a long-tail problem,
where the challenge comes from a small number of very specific cases. We investigate the impact of Safety
RLHF by taking two intermediate Llama 2-Chat checkpoints—one without adversarial prompts in the RLHF
stage and one with them—and score their responses on our test sets using our safety and helpfulness reward
models. In Figure 14, we plot the score distribution shift of the safety RM on the safety test set (left) and that
of the helpfulness RM on the helpfulness test set (right). In the left hand side of the figure, we observe that
the distribution of safety RM scores on the safety set shifts to higher reward scores after safety tuning with
RLHF, and that the long tail of the distribution near zero thins out. A clear cluster appears on the top-left
corner suggesting the improvements of model safety. On the right side, we do not observe any gathering
pattern below the y = x line on the right hand side of Figure 14, which indicates that the helpfulness score
distribution is preserved after safety tuning with RLHF. Put another way, given sufficient helpfulness training
data, the addition of an additional stage of safety mitigation does not negatively impact model performance
on helpfulness to any notable degradation. A qualitative example is shown in Table 12.
Impact of Safety Data Scaling.
A tension between helpfulness and safety of LLMs has been observed in
previous studies (Bai et al., 2022a). To better understand how the addition of safety training data affects
general model performance, especially helpfulness, we investigate the trends in safety data scaling by
adjusting the amount of safety data used in the RLHF stage.<br>

**Node ID:** e6cd2ddb-1c66-4476-ab75-349e4da18c1c<br>**Similarity:** 0.8768680873899801<br>**Text:** We evaluate them using our safety and helpfulness reward models described in Section 3.2.2. For
24


0.0
0.2
0.4
0.6
0.8
1.0
Safety RM Score before Safety RLHF
0.0
0.2
0.4
0.6
0.8
1.0
Safety RM Score after Safety RLHF
Safety 
 Improvement
0
1000
0
1000
0.0
0.2
0.4
0.6
0.8
1.<br>

**Node ID:** bc99d4c1-270a-4a40-b971-1727ab26ab65<br>**Similarity:** 0.8703461945378743<br>**Text:** This teaches
the model to align with our safety guidelines even before RLHF, and thus lays the foundation for
high-quality human preference data annotation.
2. Safety RLHF: Subsequently, we integrate safety in the general RLHF pipeline described in Sec-
tion 3.2.2. This includes training a safety-specific reward model and gathering more challenging
adversarial prompts for rejection sampling style fine-tuning and PPO optimization.
3. Safety Context Distillation: Finally, we refine our RLHF pipeline with context distillation (Askell
et al., 2021b).<br>

In [20]:
for node in base_nodes:
    display_source_node(node, source_length=10000)

**Node ID:** b0950ac0-083d-48ab-bdca-14b686549567<br>**Similarity:** 0.8982939331758837<br>**Text:** A qualitative example is shown in Table 12.
Impact of Safety Data Scaling.
A tension between helpfulness and safety of LLMs has been observed in
previous studies (Bai et al., 2022a). To better understand how the addition of safety training data affects
general model performance, especially helpfulness, we investigate the trends in safety data scaling by
adjusting the amount of safety data used in the RLHF stage.<br>

**Node ID:** 4ec27489-f9c4-4642-975c-69e6c279d6a6<br>**Similarity:** 0.8900448651894107<br>**Text:** A clear cluster appears on the top-left
corner suggesting the improvements of model safety. On the right side, we do not observe any gathering
pattern below the y = x line on the right hand side of Figure 14, which indicates that the helpfulness score
distribution is preserved after safety tuning with RLHF. Put another way, given sufficient helpfulness training
data, the addition of an additional stage of safety mitigation does not negatively impact model performance
on helpfulness to any notable degradation. A qualitative example is shown in Table 12.
Impact of Safety Data Scaling.<br>

**Node ID:** 754e8c80-0739-4b73-9c4f-b49aed28d11a<br>**Similarity:** 0.8893585339004003<br>**Text:** 0
0.2
0.4
0.6
0.8
1.0
Helpfulness RM Score before Safety RLHF
0.0
0.2
0.4
0.6
0.8
1.0
Helpfulness RM Score after Safety RLHF
0
1000
0
1000
Figure 14: Impact of safety RLHF measured by reward model score distributions. Left: safety reward
model scores of generations on the Meta Safety test set. The clustering of samples in the top left corner
suggests the improvements of model safety.<br>

**Node ID:** 504ae267-b6ab-4479-900a-5d7384de2f68<br>**Similarity:** 0.8794657077807086<br>**Text:** In Figure 14, we plot the score distribution shift of the safety RM on the safety test set (left) and that
of the helpfulness RM on the helpfulness test set (right). In the left hand side of the figure, we observe that
the distribution of safety RM scores on the safety set shifts to higher reward scores after safety tuning with
RLHF, and that the long tail of the distribution near zero thins out. A clear cluster appears on the top-left
corner suggesting the improvements of model safety.<br>

**Node ID:** e6cd2ddb-1c66-4476-ab75-349e4da18c1c<br>**Similarity:** 0.8768680873899801<br>**Text:** We evaluate them using our safety and helpfulness reward models described in Section 3.2.2. For
24


0.0
0.2
0.4
0.6
0.8
1.0
Safety RM Score before Safety RLHF
0.0
0.2
0.4
0.6
0.8
1.0
Safety RM Score after Safety RLHF
Safety 
 Improvement
0
1000
0
1000
0.0
0.2
0.4
0.6
0.8
1.<br>

**Node ID:** bc99d4c1-270a-4a40-b971-1727ab26ab65<br>**Similarity:** 0.8703461945378743<br>**Text:** This teaches
the model to align with our safety guidelines even before RLHF, and thus lays the foundation for
high-quality human preference data annotation.
2. Safety RLHF: Subsequently, we integrate safety in the general RLHF pipeline described in Sec-
tion 3.2.2. This includes training a safety-specific reward model and gathering more challenging
adversarial prompts for rejection sampling style fine-tuning and PPO optimization.
3. Safety Context Distillation: Finally, we refine our RLHF pipeline with context distillation (Askell
et al., 2021b).<br>

In [21]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine(retriever=base_retriever)

response = query_engine.query(query_str)

print(str(response))

Adjusting the amount of safety data in the RLHF stage can impact the model's safety and helpfulness. The distribution of safety reward model scores on the safety set shifts to higher reward scores, and the long tail of the distribution near zero thins out. A cluster appears, which suggests improvements in model safety. Also, the helpfulness score distribution is preserved, and the addition of a safety mitigation stage does not negatively impact model performance on helpfulness.

